# Notebook 3: Model Evaluation
**ANPR System – MIPA UGM Parking Lot**

Evaluates all three YOLOv8 variants (n/s/m) on the test set:
- mAP@0.5, mAP@0.5:0.95, Precision, Recall, F1
- PR curves and confusion matrices
- Inference speed (FPS) on CPU and GPU
- Visual detection results on sample images

## 1. Setup

In [ ]:
!pip install -q ultralytics seaborn
from google.colab import drive
drive.mount('/content/drive')

import torch
print(f'CUDA: {torch.cuda.is_available()}')

DRIVE_ROOT = '/content/drive/MyDrive/ANPR_MIPA_UGM'
DATA_YAML = f'{DRIVE_ROOT}/processed/data.yaml'
WEIGHTS = {
    'yolov8n': f'{DRIVE_ROOT}/weights/best_yolov8n.pt',
    'yolov8s': f'{DRIVE_ROOT}/weights/best_yolov8s.pt',
    'yolov8m': f'{DRIVE_ROOT}/weights/best_yolov8m.pt',
}

## 2. Validate All Variants on Test Set

In [ ]:
from ultralytics import YOLO
import os

all_metrics = {}

for variant, weight_path in WEIGHTS.items():
    if not os.path.exists(weight_path):
        print(f'{variant}: weights not found, skipping')
        continue
    
    print(f'\nValidating {variant}...')
    model = YOLO(weight_path)
    results = model.val(data=DATA_YAML, split='test', verbose=True)
    
    mp = float(results.box.mp)
    mr = float(results.box.mr)
    f1 = 2 * mp * mr / max(mp + mr, 1e-6)
    
    all_metrics[variant] = {
        'mAP50': float(results.box.map50),
        'mAP50_95': float(results.box.map),
        'precision': mp,
        'recall': mr,
        'f1': f1,
    }
    print(f'  mAP@0.5={all_metrics[variant]["mAP50"]:.4f}')

print('\nAll metrics collected.')

## 3. Comparison Table

In [ ]:
import pandas as pd

df = pd.DataFrame(all_metrics).T
df.index.name = 'Variant'
df = df.round(4)
print(df.to_string())

# Highlight best values
display(df.style.highlight_max(axis=0, color='lightgreen').format('{:.4f}'))

## 4. Bar Chart Comparison

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

variants = list(all_metrics.keys())
metric_names = ['mAP50', 'mAP50_95', 'precision', 'recall', 'f1']
labels = ['mAP@0.5', 'mAP@0.5:0.95', 'Precision', 'Recall', 'F1']
colors = ['#4c72b0', '#55a868', '#dd8452', '#c44e52', '#8172b3']

x = np.arange(len(variants))
width = 0.15

fig, ax = plt.subplots(figsize=(12, 6))
for i, (metric, label, color) in enumerate(zip(metric_names, labels, colors)):
    vals = [all_metrics[v][metric] for v in variants]
    bars = ax.bar(x + i * width, vals, width, label=label, color=color)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f'{val:.3f}', ha='center', va='bottom', fontsize=7, rotation=90)

ax.set_xlabel('Model Variant')
ax.set_ylabel('Score')
ax.set_title('YOLOv8 Variant Comparison on Test Set')
ax.set_xticks(x + width * 2)
ax.set_xticklabels(variants)
ax.legend(loc='lower right')
ax.set_ylim(0, 1.15)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(f'{DRIVE_ROOT}/detection_comparison.png', dpi=150)
plt.show()

## 5. Inference Speed (FPS)

In [ ]:
import time
import cv2
import glob

test_images = glob.glob(f'{DRIVE_ROOT}/processed/images/test/*.jpg')[:50]

speed_results = {}

for variant, weight_path in WEIGHTS.items():
    if not os.path.exists(weight_path):
        continue
    model = YOLO(weight_path)
    
    # CPU
    model.to('cpu')
    t0 = time.time()
    for img_path in test_images:
        model.predict(img_path, verbose=False)
    cpu_fps = len(test_images) / (time.time() - t0)
    
    # GPU (if available)
    gpu_fps = None
    if torch.cuda.is_available():
        model.to('cuda')
        # Warmup
        for img_path in test_images[:5]:
            model.predict(img_path, verbose=False)
        t0 = time.time()
        for img_path in test_images:
            model.predict(img_path, verbose=False)
        gpu_fps = len(test_images) / (time.time() - t0)
    
    speed_results[variant] = {'cpu_fps': cpu_fps, 'gpu_fps': gpu_fps}
    print(f'{variant}: CPU={cpu_fps:.1f} FPS', end='')
    if gpu_fps:
        print(f', GPU={gpu_fps:.1f} FPS', end='')
    print()

## 6. Visual Detection Results

In [ ]:
import cv2
import matplotlib.pyplot as plt

# Use the best variant (yolov8s) for visual results
model = YOLO(WEIGHTS.get('yolov8s', list(WEIGHTS.values())[0]))
sample_images = test_images[:6]

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, img_path in zip(axes.flat, sample_images):
    result = model.predict(img_path, verbose=False, conf=0.5)[0]
    img_with_boxes = result.plot()
    img_rgb = cv2.cvtColor(img_with_boxes, cv2.COLOR_BGR2RGB)
    ax.imshow(img_rgb)
    n_boxes = len(result.boxes)
    ax.set_title(f'{n_boxes} plate(s) detected', fontsize=9)
    ax.axis('off')

plt.suptitle('YOLOv8s Detection Results on Test Set', fontsize=14)
plt.tight_layout()
plt.savefig(f'{DRIVE_ROOT}/detection_samples.png', dpi=150)
plt.show()